# [심화] 트랜잭션 - 재고 차감 + 주문 추가

`sql/02_transaction.sql`의 `place_order` 함수를 RPC로 호출합니다.
재고 차감과 주문 추가가 하나의 트랜잭션이라, 중간에 실패하면 둘 다 취소(ROLLBACK)됩니다.

In [1]:
import sys
sys.path.append("..")  # 프로젝트 루트의 shopping 패키지를 불러오기 위해

from pprint import pprint
from postgrest.exceptions import APIError
from shopping import users, products, orders

In [2]:
hong = users.get_user_by_email("hong@example.com")
mouse = products.get_products()[1]
print("주문 전 재고:", mouse["name"], mouse["stock"])

주문 전 재고: 인체공학 버티컬 마우스 20


## 1. 성공 - COMMIT

In [3]:
result = orders.place_order(hong["id"], mouse["id"], 3)
pprint(result)
print("주문 후 재고:", products.get_product(mouse["id"])["stock"])

{'order_id': 'f7a49cd4-721b-48ba-91de-a2914bdb8c90',
 'success': True,
 'total_price': 165000}
주문 후 재고: 17


## 2. 실패 - 재고 부족

In [4]:
before = products.get_product(mouse["id"])["stock"]
try:
    orders.place_order(hong["id"], mouse["id"], 999)
except APIError as e:
    print("주문 실패:", e.message)
print("재고 변화 없음:", before, "->", products.get_product(mouse["id"])["stock"])

주문 실패: 주문 처리 실패 (트랜잭션 롤백): 재고가 부족합니다. (재고: 17, 주문: 999)


재고 변화 없음: 17 -> 17


## 3. 실패 - 재고 차감 후 주문 추가 단계에서 FK 위반 → 재고 차감까지 ROLLBACK

In [5]:
before = products.get_product(mouse["id"])["stock"]
try:
    orders.place_order("00000000-0000-0000-0000-000000000000", mouse["id"], 1)
except APIError as e:
    print("주문 실패:", e.message)
print("재고 변화 없음:", before, "->", products.get_product(mouse["id"])["stock"])

주문 실패: 주문 처리 실패 (트랜잭션 롤백): insert or update on table "orders" violates foreign key constraint "orders_user_id_fkey"
재고 변화 없음: 17 -> 17


## 정리 - 테스트 주문 삭제, 재고 원복

In [6]:
for o in orders.get_orders(hong["id"]):
    orders.delete_order(o["id"])
products.update_product(mouse["id"], price=mouse["price"], stock=mouse["stock"])
print("정리 후 재고:", products.get_product(mouse["id"])["stock"])

정리 후 재고: 20
